In [1]:
import os
import joblib
import warnings
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

In [2]:
# -----------------------------
# Try importing XGBoost
# -----------------------------
try:
    from xgboost import XGBClassifier
    xgb_available = True
except:
    xgb_available = False


# ============================
# Create models folder
# ============================

os.makedirs("models", exist_ok=True)

In [3]:
# Function to train models
# ============================

def train_dataset(dataset_name,
                  csv_path,
                  target_column,
                  model_filename,
                  scaler_filename):

    print("\n========================================")
    print(f"Training {dataset_name}")
    print("========================================")

    df = pd.read_csv(csv_path)

    # Remove duplicates
    df = df.drop_duplicates()

    # Separate X and y
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Encode categorical columns
    encoder = LabelEncoder()

    for col in X.columns:
        if X[col].dtype == object:
            X[col] = encoder.fit_transform(X[col].astype(str))

    if y.dtype == object:
        y = encoder.fit_transform(y)

    # Missing values
    imputer = SimpleImputer(strategy="median")
    X = imputer.fit_transform(X)

    # Scaling
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Train Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Models
    models = {
        "Logistic Regression":
            LogisticRegression(max_iter=1000),

        "SVM":
            SVC(kernel="rbf"),

        "Random Forest":
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
    }

    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            eval_metric="logloss",
            random_state=42
        )

    best_model = None
    best_accuracy = 0

    print("\nAccuracy Results\n")

    for name, model in models.items():

        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        acc = accuracy_score(y_test, pred)

        print(f"{name:<22}: {acc:.4f}")

        if acc > best_accuracy:
            best_accuracy = acc
            best_model = model

    # Save model
    joblib.dump(best_model, f"models/{model_filename}")

    # Save scaler
    joblib.dump(scaler, f"models/{scaler_filename}")

    print("\nBest Model Saved")
    print("Accuracy :", round(best_accuracy * 100, 2), "%")


In [4]:
# Diabetes
# ======================================================

train_dataset(
    dataset_name="Diabetes",
    csv_path="datasets/diabetes.csv",
    target_column="Outcome",
    model_filename="diabetes.pkl",
    scaler_filename="diabetes_scaler.pkl"
)



Training Diabetes

Accuracy Results

Logistic Regression   : 0.7143
SVM                   : 0.7468
Random Forest         : 0.7532
XGBoost               : 0.7403

Best Model Saved
Accuracy : 75.32 %


In [ ]:
target_column="HeartDisease"

In [ ]:
print(pd.read_csv("datasets/heart.csv").columns)

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='str')


In [ ]:
train_dataset(
    dataset_name="Heart Disease",
    csv_path="datasets/heart.csv",
    target_column="target",
    model_filename="heart.pkl",
    scaler_filename="heart_scaler.pkl"
)


Training Heart Disease

Accuracy Results

Logistic Regression   : 0.8033
SVM                   : 0.7705
Random Forest         : 0.7541
XGBoost               : 0.7213

Best Model Saved
Accuracy : 80.33 %


In [ ]:
target_column="diagnosis"

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/breast_cancer.csv")

print(df.head())
print(df.columns)
print(df["diagnosis"].unique())
print(df.dtypes)

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  perimeter_worst  area_wor

In [ ]:
print(df.columns)
print(df["diagnosis"].unique())

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst'],
      dtype='str')
<StringArray>
['M', 'B']
Length: 2, dtype: str


In [ ]:
y = df[target_column]

In [ ]:
print("Before encoding:", y.unique())

Before encoding: <StringArray>
['M', 'B']
Length: 2, dtype: str


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print("After encoding:", set(y))

After encoding: {np.int64(0), np.int64(1)}


In [ ]:
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("datasets/breast_cancer.csv")

# Encode target if it is text
if df[target_column].dtype == "object":
    encoder = LabelEncoder()
    df[target_column] = encoder.fit_transform(df[target_column])

    print("Class Mapping:")
    for i, c in enumerate(encoder.classes_):
        print(c, "->", i)

In [ ]:
X = df.drop(columns=[target_column])
y = df[target_column]

In [ ]:
df = pd.read_csv("datasets/breast_cancer.csv")

# Remove unnecessary columns
df.drop(columns=["id", "Unnamed: 32"], inplace=True, errors="ignore")

# Encode diagnosis
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df[target_column] = encoder.fit_transform(df[target_column])

X = df.drop(columns=[target_column])
y = df[target_column]

In [ ]:
df = pd.read_csv("datasets/breast_cancer.csv")
print(df.head())
print(df.columns)
print(df["diagnosis"].unique())
print(df.dtypes)

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  perimeter_worst  area_wor

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("datasets/breast_cancer.csv")

# Remove unnecessary columns
df.drop(columns=["id", "Unnamed: 32"], inplace=True, errors="ignore")

print("Before Encoding:")
print(df["diagnosis"].unique())

encoder = LabelEncoder()
df["diagnosis"] = encoder.fit_transform(df["diagnosis"])

print("\nAfter Encoding:")
print(df["diagnosis"].unique())

print("\nData Type:")
print(df["diagnosis"].dtype)

Before Encoding:
<StringArray>
['M', 'B']
Length: 2, dtype: str

After Encoding:
[1 0]

Data Type:
int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [ ]:
pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", accuracy)

Accuracy: 0.9649122807017544


In [ ]:
import os

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/cancer.pkl")
joblib.dump(scaler, "models/cancer_scaler.pkl")
joblib.dump(encoder, "models/cancer_encoder.pkl")

['models/cancer_encoder.pkl']

In [ ]:
model = joblib.load("models/cancer.pkl")
scaler = joblib.load("models/cancer_scaler.pkl")
encoder = joblib.load("models/cancer_encoder.pkl")

In [ ]:
sample = X.iloc[[0]]

sample = scaler.transform(sample)

prediction = model.predict(sample)

result = encoder.inverse_transform(prediction)

print("Prediction:", result[0])

Prediction: M
